# Lesson 13 - Agent Memory wit Cognee Knowledge Graphs


## Setup

Dis notebook dey show how to build one smart **coding assistant** wey get persistent memory by using [**Cognee**](https://www.cognee.ai/) knowledge graphs and di **Microsoft Agent Framework** (MAF).

Cognee dey turn unstructured text into one structured, queryable knowledge graph wey get vector embeddings – dis dey give your agent rich, relationship-aware long-term memory.

### Wetin You Go Learn
1. **Build Knowledge Graphs**: Turn developer profiles and best practices into structured, queryable knowledge.
2. **Integrate Cognee with MAF**: Use `@tool` functions so dat MAF agent fit query Cognee knowledge graph.
3. **Session-Aware Conversations**: Keep context across multiple questions for di same session.
4. **Long-Term Memory**: Save important knowledge across sessions and fit bring am out for new conversations.

### Prerequisites
- Python 3.9+
- Redis wey dey run locally (`docker run -d -p 6379:6379 redis`) for session management
- One LLM API key (like OpenAI) — set `LLM_API_KEY` for `.env`
- `CACHING=true` for `.env` (dis one mandatory for Cognee sessions)
- Azure AI Foundry project wey get deployed chat model
- `AZURE_AI_PROJECT_ENDPOINT` and `AZURE_AI_MODEL_DEPLOYMENT_NAME` for `.env`
- Azure CLI wey you don authenticate (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Types of Agent Memory

Dis notebook dey explore di same three memory types from di main Lesson 13 notebook, but e dey use Cognee as di long-term memory backend:

| Memory Type | Mechanism | Lifetime |
|---|---|---|
| **Working** | `agent.create_session()` (MAF) | Single conversation |
| **Short-term** | Cognee session cache (Redis) | Single session |
| **Long-term** | Cognee knowledge graph + vectors | Permanent |

### Cognee's Memory Architecture
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Prepare Cognee Storage


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Part 1 — Building di Knowledge Base

We dey take three kain data comot to build one thorough knowledge base for our coding assistant:

1. **Developer Profile** — personal expertise and technical background
2. **Python Best Practices** — di Zen of Python wit practical guidelines
3. **Historical Conversations** — past Q&A sessions between developers and AI assistants


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Visualize the Knowledge Graph

Cognee fit show interactive HTML visualization of the entities and relationships e comot.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Enrich Memory with Memify

`memify()` dey analyze di knowledge graph and e go create smart rules — wey go fit identify patterns, best practices, and di relationship wey dey between concepts.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Part 2 — MAF Agent wit Cognee Tools

Now we go create one MAF agent we fit query Cognee knowledge graph through `@tool` functions. Dis one go make the agent fit use the full power wey graph-aware semantic search get, and e go still keep di conversational context through sessions.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## Working Memory wit Sessions

Di `AgentSession` (wey dem create wit `agent.create_session()`) dey provide working memory inside session. Di agent fit look back earlier messages con still ask Cognee long-term knowledge graph.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## New Session — Long-Term Memory Persists

Starting fresh session dey clear working memory, but di Cognee knowledge graph still dey available. Di agent fit carry di same long-term knowledge come back for completely new conversation.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Summary

For dis notebook, you build one coding assistant wey combine **MAF kerja memory** (`agent.create_session()`) with **Cognee long-term knowledge graph**.

### Wetin You Don Learn
1. **Knowledge graph construction**: Cognee dey chop unstructured text come build graph + vector memory.
2. **Graph enrichment with memify**: Fakts wey dem derive and better relationships ontop your graph wey don dey.
3. **MAF + Cognee integration**: `@tool` functions dey allow MAF agent query Cognee graph in natural way.
4. **Working memory + long-term memory**: `AgentSession` (via `agent.create_session()`) dey give session context while Cognee dey give persistent knowledge.
5. **Filtered search with NodeSets**: E dey allow target specific parts of the knowledge graph (for example na only principles).

### Key Takeaways
- **Cognee** dey turn raw text to structured, relationship-aware memory — e strong pass flat vector store.
- **`@tool` functions** dey connect MAF agents and external knowledge systems nicely.
- **`AgentSession`** (via `agent.create_session()`) dey keep per-conversation context separate from long-lasting knowledge.
- The same knowledge graph dey serve different sessions and agents.

### Real-World Applications
- **Developer copilots**: Code review, incident analysis, architecture assistants
- **Customer-facing copilots**: Support agents wey dey use product docs, FAQs, and CRM notes
- **Internal expert copilots**: Policy, legal, or security assistants wey dey reason over guidelines
- **Unified data layers**: Dem fit join structured and unstructured data as one queryable graph

### Next Steps
- Try experiment with temporal awareness for Cognee
- Define OWL ontology for domain-specific graph quality
- Add user feedback loops to improve how retrieval dey work over time
- Scale to multi-agent systems wey dey share the same Cognee memory layer


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dis document don translate wit AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator). Even if we dey try make am correct, abeg know say machine translation fit get error or wahala. Di original document wey dem write for di correct language na di real correct one. For important info, e better make human professional translate am. We no go responsible if person no understand or misinterpret di message wey dis translation show.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
